# 05 - GAT-GCN Price Prediction
Modelo de atención sobre grafos para precios de venta en CABA.

In [32]:

# Instalación opcional (si el kernel no tiene deps)
# !pip install -r ../requirements.txt
# !pip install -r ../ml_core/requirements.txt


In [33]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path(os.path.abspath(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import torch

from ml_core.models import GraphAttentionGCN
from ml_core.preprocessing import KNHS


In [34]:
# Paths
data_path = '../scraper_service/storage/data/arg_venta_caba_processed.csv'

# Hyperparámetros de grafo
k_neighbors = 15  # vecinos por nodo para construir aristas
radius_km = 3.0
bandwidth_km = 2.0  # para pesos locales (LGWR aproximada)


In [35]:

# Cargar datos procesados

df = pd.read_csv(data_path)
print(df.shape)
df.head()


(10614, 31)


,idx,id,url,precio,moneda,expensas,tipo_unidad,area_m2_cubierta,area_m2_descubierta,area_m2_total,...,valido_desde,valido_hasta,cocheras,dormitorios,index_right,barrio,comuna,estado_num,log_precio,precio_sobre_m2
0,4257,18875423.0,https://www.argenprop.com/departamento-en-vent...,70000.0,USD,113400.0,NaN,34.0,0.00,34.0,...,2026-02-05 15:18:35.864925,NaN,0.0,NaN,35.0,Villa Crespo,15.0,4.0,11.156251,2058.823529
1,4262,18875458.0,https://www.argenprop.com/departamento-en-vent...,284000.0,USD,0.0,Duplex,123.0,49.54,196.0,...,2026-02-05 15:19:02.071369,NaN,0.0,NaN,36.0,Villa Del Parque,11.0,5.0,12.556730,1448.979592
2,4265,18172428.0,https://www.argenprop.com/departamento-en-vent...,125000.0,USD,230000.0,NaN,84.0,0.00,84.0,...,2026-02-05 15:19:16.333598,NaN,0.0,NaN,7.0,Caballito,6.0,3.0,11.736069,1488.095238
3,4266,8369836.0,https://www.argenprop.com/departamento-en-vent...,69500.0,USD,105000.0,Departamento,45.0,0.00,45.0,...,2026-02-05 15:19:22.322805,NaN,0.0,NaN,7.0,Caballito,6.0,4.0,11.149082,1544.444444
4,4267,15860074.0,https://www.argenprop.com/departamento-en-vent...,84900.0,USD,110000.0,Departamento,63.0,0.00,63.0,...,2026-02-05 15:19:26.923873,NaN,0.0,NaN,12.0,Flores,7.0,5.0,11.349229,1347.619048


In [36]:
# Filtrar outliers extremos en log_precio usando IQR robusto
q1, q3 = df['log_precio'].quantile([0.01, 0.99])
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
df = df[(df['log_precio'] >= low) & (df['log_precio'] <= high)].reset_index(drop=True)
print(f"Filtrado outliers log_precio: rango [{low:.3f}, {high:.3f}], n={len(df)}")


Filtrado outliers log_precio: rango [5.705, 18.394], n=10605


In [37]:

# Selección simple de features numéricas (excluimos targets y columnas no numéricas)

feature_cols = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'expensas',
    'antiguedad',
    'banos',
    'cocheras',
    'latitud',
    'longitud',
]

# Eliminar filas con NaN en target o features seleccionadas
df = df.dropna(subset=feature_cols + ['log_precio']).reset_index(drop=True)

y = df['log_precio'].astype(float).values
X = df[feature_cols].copy()

print(f"Usando {len(feature_cols)} features")
print(f"Dataset tras dropna: {len(df)} filas")


Usando 9 features
Dataset tras dropna: 9163 filas


In [38]:

# Split train / test
idx = np.arange(len(df))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train, y_train = X.iloc[train_idx], y[train_idx]
X_test, y_test = X.iloc[test_idx], y[test_idx]

coords_train = np.deg2rad(df.loc[train_idx, ['latitud', 'longitud']].values)
coords_test = np.deg2rad(df.loc[test_idx, ['latitud', 'longitud']].values)


In [39]:

# Estandarizar features
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index,
)


In [40]:
# Pesos locales por feature (LGWR aproximada sobre train) y proyección a test

def compute_local_feature_weights(X_df, y_array, coords_rad, bandwidth_km=2.0, k_neighbors=50):
    """Estima pesos por feature usando regresión local ponderada espacialmente."""
    from sklearn.neighbors import BallTree

    X = X_df.to_numpy()
    n, d = X.shape
    tree = BallTree(coords_rad, metric='haversine')
    weights = np.zeros((n, d), dtype=float)

    for i in range(n):
        dist, idx = tree.query(coords_rad[i:i+1], k=min(k_neighbors + 1, n))
        dist = dist.ravel() * 6371.0
        idx = idx.ravel()
        mask = dist > 0
        dist, idx = dist[mask], idx[mask]
        if len(idx) == 0:
            weights[i] = 1.0
            continue

        kernel = np.exp(-(dist ** 2) / (bandwidth_km ** 2))
        Xn = X[idx]
        wn = kernel[:, None]
        A = Xn * wn
        AtX = A.T @ Xn + np.eye(d) * 1e-6  # regularización numérica
        AtY = A.T @ y_array[idx]
        beta = np.linalg.solve(AtX, AtY)
        weights[i] = np.abs(beta)

    return weights

weights_train = compute_local_feature_weights(
    X_train,
    y_train,
    coords_train,
    bandwidth_km=bandwidth_km,
    k_neighbors=k_neighbors,
)

# Proyectar pesos a test como promedio ponderado de vecinos de train
_tree_w = BallTree(coords_train, metric='haversine')
k_proj = min(k_neighbors, len(train_idx))
dist_proj, idx_proj = _tree_w.query(coords_test, k=k_proj)
kernel_proj = np.exp(-((dist_proj * 6371.0) ** 2) / (bandwidth_km ** 2))
kernel_proj = kernel_proj / (kernel_proj.sum(axis=1, keepdims=True) + 1e-9)
weights_test = (kernel_proj[..., None] * weights_train[idx_proj]).sum(axis=1)

# Columnas de pesos asociadas a cada feature
weight_cols = [f'w_{c}' for c in feature_cols]


In [41]:
# Construir grafo para train usando KNHS con distancia localmente pesada

graph_train = X_train.copy().reset_index(drop=True)
graph_train['lat_deg'] = np.rad2deg(coords_train[:, 0])
graph_train['lon_deg'] = np.rad2deg(coords_train[:, 1])
for col, vals in zip(weight_cols, weights_train.T):
    graph_train[col] = vals

builder = KNHS(
    lat_col='lat_deg',
    lon_col='lon_deg',
    feature_cols=feature_cols,
    weight_cols=weight_cols,
    distance='local_weighted',
    radius_km=radius_km,
    k=k_neighbors,
    add_reverse=True,
)

edge_index_train, edge_attr_train = builder.build(graph_train)
print('Aristas train:', edge_index_train.shape[1])


Aristas train: 219900


In [42]:
# Grafo para test (inductivo) usando los pesos proyectados

graph_test = X_test.copy().reset_index(drop=True)
graph_test['lat_deg'] = np.rad2deg(coords_test[:, 0])
graph_test['lon_deg'] = np.rad2deg(coords_test[:, 1])
for col, vals in zip(weight_cols, weights_test.T):
    graph_test[col] = vals

edge_index_test, edge_attr_test = builder.build(graph_test)
print('Aristas test:', edge_index_test.shape[1])


Aristas test: 54990


In [43]:
# Búsqueda más exhaustiva de hiperparámetros (grid)
from itertools import product

hidden_opts = [64, 96, 128]
head_opts = [2, 4]
dropout_opts = [0.05, 0.10]
lr_opts = [1e-3, 5e-4]

configs = [
    dict(hidden=h, num_heads=nh, dropout=do, lr=lr)
    for h, nh, do, lr in product(hidden_opts, head_opts, dropout_opts, lr_opts)
]

results = []
for cfg in configs:
    # saltar combinaciones donde hidden no divide en cabezas (por seguridad)
    if cfg['hidden'] % cfg['num_heads'] != 0:
        continue
    print("Entrenando config", cfg)
    model_t = GraphAttentionGCN(
        feature_names=feature_cols,
        edge_dim=2,
        hidden=cfg['hidden'],
        num_layers=2,
        num_heads=cfg['num_heads'],
        dropout=cfg['dropout'],
        lr=cfg['lr'],
        weight_decay=1e-4,
        patience=50,
    )
    _ = model_t.fit(
        X_train,
        y_train,
        edge_index=edge_index_train,
        edge_attr=edge_attr_train,
        epochs=400,
    )
    preds = model_t.predict(
        X_test,
        edge_index=edge_index_test,
        edge_attr=edge_attr_test,
    )
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    mape = np.mean(np.abs((np.exp(preds) - np.exp(y_test)) / (np.exp(y_test) + 1e-8))) * 100
    results.append({**cfg, "rmse": rmse, "mae": mae, "mape": mape})

results_df = pd.DataFrame(results).sort_values(["mae", "rmse"]).reset_index(drop=True)
best_config = results_df.iloc[0].to_dict()
print("Resultados (ordenado por MAE, RMSE):")
print(results_df)
print("Mejor config:", best_config)


Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 64, 'num_heads': 4, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.001}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.05, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.001}
Entrenando config {'hidden': 96, 'num_heads': 2, 'dropout': 0.1, 'lr': 0.0005}
Entrenando config {'hidden': 96, 'num_heads': 4, 'dr

In [62]:
# Inicializar modelo con la mejor config encontrada
edge_dim = 2  # distancia km + distancia de features (localmente pesada)
model = GraphAttentionGCN(
    feature_names=feature_cols,
    edge_dim=edge_dim,
    hidden=int(best_config['hidden']),
    num_layers=2,
    num_heads=int(best_config['num_heads']),
    dropout=float(best_config['dropout']),
    lr=float(best_config['lr']),
    weight_decay=1e-4,
    patience=50,
)


In [63]:

# Entrenamiento
_ = model.fit(
    X_train,
    y_train,
    edge_index=edge_index_train,
    edge_attr=edge_attr_train,
    epochs=6000,
)


Early stopping en epoch 1232, best loss=0.298456


In [64]:
# Predicción en test (usando grafo de test)

y_pred = model.predict(
    X_test,
    edge_index=edge_index_test,
    edge_attr=edge_attr_test,
)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

# MAPE en escala de precio (deshacemos el log)
price_true = np.exp(y_test)
price_pred = np.exp(y_pred)
mape = np.mean(np.abs((price_pred - price_true) / (price_true + 1e-8))) * 100

print(f"RMSE log-precio: {rmse:.4f}")
print(f"MAE log-precio: {mae:.4f}")
print(f"MAPE precio: {mape:.2f}%")


RMSE log-precio: 0.5478
MAE log-precio: 0.3252
MAPE precio: 32.67%


In [65]:

# Guardar resultados rápidos
results = pd.DataFrame({
    'true': y_test,
    'pred': y_pred,
    'error': y_pred - y_test,
})
results.head()


,true,pred,error
0,12.560244,12.259728,-0.300516
1,11.512925,11.507135,-0.005790
2,12.601487,12.655640,0.054152
3,11.608236,11.664396,0.056161
4,10.618885,11.055572,0.436686



## Notas
- Este cuaderno entrena y evalúa de forma simple; puedes ajustar `k_neighbors`, `hidden`, `num_layers`, etc.
- Si quieres predecir precio en escala original, usa `np.exp(pred)`.
- Para usar el grafo completo con entrenamiento transductivo, reconstruye `edge_index` con todos los nodos y ajusta las máscaras de train/test.
